# Chapter 12 — Arbitrage Pricing with Term Structure Models

教材 pp. 148–156；PDF 页序为教材页码加 11。



下列未明确标为教材例题的数据均为合成验证数据，不声称复现教材中的原始市场样本。

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from numpy.testing import assert_allclose
from tree_calculator import *

教材图12.1与式(12.7)，pp.148–150：反推风险中性概率及债券价格。

In [2]:
up=1000/(1+.025/2)
down=1000/(1+.015/2)
market=1000/(1+.0215/2)**2
p=risk_neutral_probability(market,up,down,.02,.5)
assert_allclose(p,.8009,atol=.0001)
rates=pd.DataFrame([[.02,.015],[np.nan,.025]])
probabilities=pd.DataFrame([[p,.5],[np.nan,.5]])
tree=BinomialRateTree(rates,probabilities,.5)
terminal=pd.DataFrame({'payoff':[1000.,1000.,1000.]})
assert_allclose(tree.price(terminal,'payoff'),market)
assert_allclose(tree.option_adjusted_spread(terminal,'payoff',market,(-.01,.1)),0,atol=1e-10)
print(p,market)

0.8009019598798706 978.8417842352513


复制价格与风险中性价格一致，归因加总恒等式；pp.149,155。

In [3]:
payoffs=pd.DataFrame({'bond':[up,down],'cash':[1.01,1.01],'option':[0.,down-990]})
prices=pd.DataFrame({'price':[market,1.]})
out=replicating_portfolio(payoffs,['bond','cash'],'option',prices,'price')
assert_allclose(out['price'],(1-p)*(down-990)/1.01)
parts=pnl_attribution(100,101,99,98,2)
assert_allclose(parts.iloc[:3].sum(),parts['total'])

## 多期曲线校准及可提前行权回溯；pp.151–155

In [4]:
rates=pd.DataFrame([[.02,.015,.01],[np.nan,.025,.02],[np.nan,np.nan,.03]])
known=pd.DataFrame([[.8,.65,.5],[np.nan,.65,.5],[np.nan,np.nan,.5]])
original=BinomialRateTree(rates,known,.5)
discounts=[]
for n in [1,2,3]:
    sub=BinomialRateTree(rates.iloc[:,:n],known.iloc[:,:n],.5)
    discounts.append(sub.price(pd.DataFrame({'payoff':np.ones(n+1)}),'payoff'))
calibrated=BinomialRateTree.calibrate_probabilities(rates,pd.DataFrame({'df':discounts}),'df',.5)
assert_allclose(calibrated.probabilities[0,0],.8)
assert_allclose(calibrated.probabilities[:2,1],.65)
terminal=pd.DataFrame({'payoff':np.ones(4)})
european=original.price(terminal,'payoff')
american=original.price(terminal,'payoff',exercise=lambda t,v:np.maximum(v,.99))
assert american>=european
assert_allclose(cmt_payments(pd.DataFrame({'y':[.025,.015]}),'y',1e6,.02,.5).payment,[2500,-2500])